## Data loading and validation

In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv(
    "../Data/Processed/BankChurners_Clean.csv",
    dtype={"customer_id": "string"}
)

df.head(10)

,customer_id,attrition_status,customer_age,gender,dependent_count,education_level,marital_status,income_category,card_category,customer_tenure_months,...,contacts_count_12_mon,credit_limit,total_revolving_bal,avg_open_to_buy,total_amt_chng_q4_q1,total_trans_amt,total_trans_ct,total_ct_chng_q4_q1,avg_utilization_ratio,churned
0,768805383,Existing Customer,45,M,3,High School,Married,$60K - $80K,Blue,39,...,3,12691.0,777,11914.0,1.335,1144,42,1.625,0.061,0
1,818770008,Existing Customer,49,F,5,Graduate,Single,Less than $40K,Blue,44,...,2,8256.0,864,7392.0,1.541,1291,33,3.714,0.105,0
2,713982108,Existing Customer,51,M,3,Graduate,Married,$80K - $120K,Blue,36,...,0,3418.0,0,3418.0,2.594,1887,20,2.333,0.000,0
3,769911858,Existing Customer,40,F,4,High School,Unknown,Less than $40K,Blue,34,...,1,3313.0,2517,796.0,1.405,1171,20,2.333,0.760,0
4,709106358,Existing Customer,40,M,3,Uneducated,Married,$60K - $80K,Blue,21,...,0,4716.0,0,4716.0,2.175,816,28,2.500,0.000,0
5,713061558,Existing Customer,44,M,2,Graduate,Married,$40K - $60K,Blue,36,...,2,4010.0,1247,2763.0,1.376,1088,24,0.846,0.311,0
6,810347208,Existing Customer,51,M,4,Unknown,Married,$120K +,Gold,46,...,3,34516.0,2264,32252.0,1.975,1330,31,0.722,0.066,0
7,818906208,Existing Customer,32,M,0,High School,Unknown,$60K - $80K,Silver,27,...,2,29081.0,1396,27685.0,2.204,1538,36,0.714,0.048,0
8,710930508,Existing Customer,37,M,3,Uneducated,Single,$60K - $80K,Blue,36,...,0,22352.0,2517,19835.0,3.355,1350,24,1.182,0.113,0
9,719661558,Existing Customer,48,M,2,Graduate,Single,$80K - $120K,Blue,36,...,3,11656.0,1677,9979.0,1.524,1441,32,0.882,0.144,0


In [4]:
print(df.shape)

(10127, 22)


In [5]:
churn_counts = df["churned"].value_counts().sort_index()
churn_rates = df["churned"].value_counts(normalize=True).sort_index()

print(churn_counts)
print()
print(churn_rates.round(2))

churned
0    8500
1    1627
Name: count, dtype: int64

churned
0    0.84
1    0.16
Name: proportion, dtype: float64


In [6]:
relationship_table = pd.crosstab(
    df["total_relationship_count"],
    df["churned"]
)

relationship_table

churned,0,1
total_relationship_count,,
1,677,233
2,897,346
3,1905,400
4,1687,225
5,1664,227
6,1670,196


In [7]:
from scipy.stats import chi2_contingency

In [8]:
chi2, p_value, dof, expected = chi2_contingency(relationship_table)

print(f"Chi-square statistic: {chi2:.2f}")
print(f"P-value: {p_value:.6f}")
print(f"Degrees of freedom: {dof}")

Chi-square statistic: 284.07
P-value: 0.000000
Degrees of freedom: 5


** The chi-square test found a statistically significant association between relationship count and churn. The P-value is also extremely low. This supports the evidence gathered from exploratory analysis.

In [9]:
n = relationship_table.to_numpy().sum()

cramers_v = np.sqrt(
    chi2 / (n * min(relationship_table.shape[0] - 1,
                    relationship_table.shape[1] - 1))
)

print(f"Cramér's V: {cramers_v:.3f}")

Cramér's V: 0.167


With a Cramers V of 0.167, the strength of the association between relationship count and churn is small, despite being statistically significant. It should be considered alongside other contributing variables.

# Hypothesis 1: Customer Engagement and Attrition

*Null hypothesis: Transaction activity does not differ between retained and attrited customers.

*Alternative hypothesis: Transaction activity differs between retained and attrited customers.

In [10]:
retained_transactions = df.loc[
    df["churned"] == 0,
    "total_trans_ct"
]

attrited_transactions = df.loc[
    df["churned"] == 1,
    "total_trans_ct"
]

print("Retained customers:")
print(retained_transactions.describe())

print("\nAttrited customers:")
print(attrited_transactions.describe())

Retained customers:
count    8500.000000
mean       68.672588
std        22.919011
min        11.000000
25%        54.000000
50%        71.000000
75%        82.000000
max       139.000000
Name: total_trans_ct, dtype: float64

Attrited customers:
count    1627.000000
mean       44.933620
std        14.568429
min        10.000000
25%        37.000000
50%        43.000000
75%        51.000000
max        94.000000
Name: total_trans_ct, dtype: float64


The data shows that retained customers demonstrate significantly higher transaction activity than attrited customers

# Independent samples Welch's T-test

A Welch's T-test is chosen because it does not assume that both groups have equal variance.

In [11]:
from scipy.stats import ttest_ind

t_stat, p_value = ttest_ind(
    retained_transactions,
    attrited_transactions,
    equal_var=False
)

print(f"T-statistic: {t_stat:.2f}")
print(f"P-value: {p_value:.6f}")

T-statistic: 54.14
P-value: 0.000000


In [12]:
from pingouin import compute_effsize

cohens_d = compute_effsize(
    retained_transactions,
    attrited_transactions,
    eftype="cohen"
)

print(f"Cohen's D result: {cohens_d:.3f}")

Cohen's D result: 1.089


Welch's T-test and Cohen's d both confirm a large difference between the retained and attrited groups.

# Hypothesis 2: inactivity levels

Test if inactivity levels meaningfully differ between attrited and retained customers

In [13]:
retained_inactive = df.loc[
    df["churned"] == 0,
    "months_inactive_12_mon"
]

attrited_inactive = df.loc[
    df["churned"] == 1,
    "months_inactive_12_mon"
]

print("Retained customers:")
print(retained_inactive.describe())

print("\nAttrited customers:")
print(attrited_inactive.describe())

Retained customers:
count    8500.000000
mean        2.273765
std         1.016741
min         0.000000
25%         1.000000
50%         2.000000
75%         3.000000
max         6.000000
Name: months_inactive_12_mon, dtype: float64

Attrited customers:
count    1627.000000
mean        2.693301
std         0.899623
min         0.000000
25%         2.000000
50%         3.000000
75%         3.000000
max         6.000000
Name: months_inactive_12_mon, dtype: float64


In [14]:
from scipy.stats import mannwhitneyu

u_stat, p_value = mannwhitneyu(
    retained_inactive,
    attrited_inactive,
    alternative="two-sided"
)

print(f"U-stat: {u_stat:.2f}")
print(f"P-value: {p_value:.8f}")

U-stat: 5142082.00
P-value: 0.00000000


In [15]:
import pingouin as pg

inactive_test = pg.mwu(
    retained_inactive,
    attrited_inactive,
    alternative="two-sided"
)

print(inactive_test)

         U_val alternative         p_val      RBC     CLES
MWU  5142082.0   two-sided  5.423116e-67 -0.25636  0.37182


Between retained and attrited customers, months inactive differed significantly, represented by a low P-value. A modest association was present, showing attrited customers generally having higher inactivity.

# Predictive modeling

Feature selection - Customer identifiers and target fields were removed to prevent data leakage. Highly redundant account variables were also removed to allow for improved interpretability.

In [16]:
numerical_features = [
    "customer_age",
    "dependent_count",
    "customer_tenure_months",
    "total_relationship_count",
    "months_inactive_12_mon",
    "contacts_count_12_mon",
    "credit_limit",
    "total_amt_chng_q4_q1",
    "total_trans_amt",
    "total_trans_ct",
    "total_ct_chng_q4_q1",
    "avg_utilization_ratio"
]

categorical_features = [
    "gender",
    "education_level",
    "marital_status",
    "income_category",
    "card_category"
]

feature_combo = numerical_features + categorical_features

X = df[feature_combo]
y = df["churned"]

In [17]:
corr_matrix = X[numerical_features].corr()
corr_matrix.round(2)

print(corr_matrix)

                          customer_age  dependent_count  \
customer_age                  1.000000        -0.122254   
dependent_count              -0.122254         1.000000   
customer_tenure_months        0.788912        -0.103062   
total_relationship_count     -0.010931        -0.039076   
months_inactive_12_mon        0.054361        -0.010768   
contacts_count_12_mon        -0.018452        -0.040505   
credit_limit                  0.002476         0.068065   
total_amt_chng_q4_q1         -0.062042        -0.035439   
total_trans_amt              -0.046446         0.025046   
total_trans_ct               -0.067097         0.049912   
total_ct_chng_q4_q1          -0.012143         0.011087   
avg_utilization_ratio         0.007114        -0.037135   

                          customer_tenure_months  total_relationship_count  \
customer_age                            0.788912                 -0.010931   
dependent_count                        -0.103062                 -0.039076  

Transaction amount and count present as moderately strong predictor relationships, but no near perfect redundancy is present

In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=1,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

print("\nTraining churn rate:")
print(y_train.value_counts(normalize=True).round(3))

print("\nTest churn rate:")
print(y_test.value_counts(normalize=True).round(3))

Training set: (8101, 17)
Test set: (2026, 17)

Training churn rate:
churned
0    0.839
1    0.161
Name: proportion, dtype: float64

Test churn rate:
churned
0    0.84
1    0.16
Name: proportion, dtype: float64


# Preprocessing

For logistic regression to work, all variables must be converted to a numeric value. Categorical variables such as gender, education level, etc will be converted.

In [19]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessing = ColumnTransformer([
    ("numerical", StandardScaler(), numerical_features),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

In [20]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("preprocessor", preprocessing),
    ("classifier", LogisticRegression(max_iter=1000, random_state=1)),
])

model.fit(X_train, y_train)
print("model trained")

model trained


In [21]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred = model.predict(X_test)

y_prob = model.predict_proba(X_test)[:, 1]

cfmt = confusion_matrix(y_test, y_pred)
print(cfmt)
print(classification_report(y_test, y_pred))

[[1634   67]
 [ 143  182]]
              precision    recall  f1-score   support

           0       0.92      0.96      0.94      1701
           1       0.73      0.56      0.63       325

    accuracy                           0.90      2026
   macro avg       0.83      0.76      0.79      2026
weighted avg       0.89      0.90      0.89      2026



The first regression model demonstrates good classification and precision. The business use case is customer prioritization rather than binary classification, so evaluation should focus on risk tiers.

In [22]:
from sklearn.metrics import roc_auc_score, average_precision_score

roc_auc = roc_auc_score(y_test, y_prob)
precision_auc = average_precision_score(y_test, y_prob)

print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC: {precision_auc:.4f}")

ROC-AUC: 0.9304
PR-AUC: 0.7496


In [23]:
test_results = pd.DataFrame({
    "actual_churn": y_test.values, "churn_probability": y_prob})

test_results = test_results.sort_values("churn_probability", ascending=False).reset_index(drop=True)

print(test_results.head(40))

    actual_churn  churn_probability
0              1           0.993103
1              1           0.988746
2              1           0.980195
3              1           0.979550
4              1           0.979230
5              1           0.972946
6              1           0.969632
7              1           0.965903
8              1           0.963701
9              1           0.955646
10             1           0.952209
11             1           0.952115
12             1           0.950977
13             1           0.949663
14             1           0.948412
15             1           0.947047
16             1           0.947037
17             1           0.946924
18             1           0.946414
19             1           0.945149
20             1           0.943208
21             1           0.942009
22             1           0.941428
23             1           0.941287
24             1           0.941219
25             1           0.940293
26             1           0

In [24]:
overall_churn_rate = test_results["actual_churn"].mean()
total_churners = test_results["actual_churn"].sum()

for pct in [0.05, 0.10, 0.15, 0.20]:

    customer_count = int(np.ceil(len(test_results) * pct))

    top_customers = test_results.head(customer_count)

    churners_captured = top_customers["actual_churn"].sum()

    precision_at_k = (
        churners_captured / customer_count
    )

    recall_k = (
        churners_captured / total_churners
    )

    lift = (
        precision_at_k / overall_churn_rate
    )

    print(f"Top {pct:.0%}")
    print(f"Customers targeted: {customer_count}")
    print(f"Churners captured: {churners_captured}")
    print(f"Precision: {precision_at_k:.3f}")
    print(f"Recall: {recall_k:.3f}")
    print(f"Lift: {lift:.2f}x")
    print()

Top 5%
Customers targeted: 102
Churners captured: 91
Precision: 0.892
Recall: 0.280
Lift: 5.56x

Top 10%
Customers targeted: 203
Churners captured: 160
Precision: 0.788
Recall: 0.492
Lift: 4.91x

Top 15%
Customers targeted: 304
Churners captured: 209
Precision: 0.688
Recall: 0.643
Lift: 4.29x

Top 20%
Customers targeted: 406
Churners captured: 250
Precision: 0.616
Recall: 0.769
Lift: 3.84x



We will target the top 10% of at risk churners for the final analysis. The model has a ~79% precision score, capturing at risk customers with high accuracy. Selecting the top 10% of highest risk customers makes the output more relevant for business use.

In [25]:
feature_names = (model.named_steps["preprocessor"].get_feature_names_out())

coefficients = (model.named_steps["classifier"].coef_[0])

coefficient_df = pd.DataFrame({"feature": feature_names,"coefficient": coefficients})

coefficient_df["feature"] = (coefficient_df["feature"].str.replace("num__", "", regex=False).str.replace("cat__", "", regex=False))

coefficient_df = coefficient_df.sort_values("coefficient",ascending=False)

print(coefficient_df)

                                        feature  coefficient
8                    numerical__total_trans_amt     1.479686
5              numerical__contacts_count_12_mon     0.567082
4             numerical__months_inactive_12_mon     0.491597
33          categorical__card_category_Platinum     0.429836
25         categorical__income_category_$120K +     0.390129
1                    numerical__dependent_count     0.155877
15       categorical__education_level_Doctorate     0.093977
18   categorical__education_level_Post-Graduate     0.090311
28    categorical__income_category_$80K - $120K     0.030478
23           categorical__marital_status_Single     0.024311
12                        categorical__gender_F     0.022459
2             numerical__customer_tenure_months    -0.005475
32              categorical__card_category_Gold    -0.013018
24          categorical__marital_status_Unknown    -0.036619
0                       numerical__customer_age    -0.085242
7               numerica

Strong indicators of reduced churn risk are high transaction count, high credit utilization, and activity represented by total_ct_chng_q4_q1.

Strong indicators of increased churn risk are high inactivity, and more customer support or bank contacts.

In [26]:
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
coefficients = model.named_steps["classifier"].coef_[0]

results = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
    "odds_change_pct": (np.exp(coefficients) - 1) * 100,
})

results = results.sort_values("odds_change_pct", ascending=False)
print(results.round(2))

                                        feature  coefficient  odds_change_pct
8                    numerical__total_trans_amt         1.48           339.16
5              numerical__contacts_count_12_mon         0.57            76.31
4             numerical__months_inactive_12_mon         0.49            63.49
33          categorical__card_category_Platinum         0.43            53.70
25         categorical__income_category_$120K +         0.39            47.72
1                    numerical__dependent_count         0.16            16.87
15       categorical__education_level_Doctorate         0.09             9.85
18   categorical__education_level_Post-Graduate         0.09             9.45
28    categorical__income_category_$80K - $120K         0.03             3.09
23           categorical__marital_status_Single         0.02             2.46
12                        categorical__gender_F         0.02             2.27
2             numerical__customer_tenure_months        -0.01    

In [27]:
retained_customers = df[df["churned"] == 0].copy()

retained_customers["churn_probability"] = model.predict_proba(
    retained_customers[X.columns]
)[:, 1]

k = int(len(retained_customers) * 0.10)

top_10_percent = (
    retained_customers
    .sort_values("churn_probability", ascending=False)
    .head(k)
    .copy()
)

viewable = top_10_percent[
    [
        "customer_id",
        "churn_probability",
        "total_trans_ct",
        "total_ct_chng_q4_q1",
        "months_inactive_12_mon",
        "total_relationship_count",
        "avg_utilization_ratio",
        "contacts_count_12_mon"
    ]
].copy()

viewable.insert(
    0,
    "risk_rank",
    range(1, len(viewable) + 1)
)

print(viewable.head(20))

viewable.to_csv(
    "../Data/Processed/Top_10_Percent_Churn_Risk.csv",
    index=False
)

      risk_rank customer_id  churn_probability  total_trans_ct  \
321           1   789270033           0.971511              13   
886           2   708655983           0.963556              24   
4179          3   797234508           0.951445              20   
510           4   713217858           0.941729              28   
152           5   712215258           0.936870              22   
1746          6   820288233           0.925574              39   
1154          7   716383683           0.925502              28   
1082          8   715511658           0.911677              27   
1915          9   710251608           0.903673              27   
163          10   714791208           0.903624              24   
1155         11   713693958           0.903112              29   
143          12   718744008           0.897935              20   
23           13   811604133           0.890737              23   
1185         14   715764483           0.885568              33   
1014      

In [ ]:
model_drivers = results[
    results["feature"].str.startswith("numerical__")
].copy()

model_drivers["feature"] = (
    model_drivers["feature"]
    .str.replace("numerical__", "", regex=False)
)

model_drivers = model_drivers[
    [
        "feature",
        "coefficient",
        "odds_change_pct"
    ]
]

model_drivers.to_csv(
    "../Data/Processed/Model_Numeric_Drivers.csv",
    index=False
)

print(model_drivers)

## Summary and key findings

A historical 16.1% attrition rate is demonstrated in the portfolio. Lower transaction activity is the strongest indicator churn that was discovered. Retained customers average 68.7 transactions, while attrited customers average 44.9. A Cohen's d of 1.09 also demonstrates a large difference. Less relationships, higher inactivity, and low credit usage are also associated with greater churn risk

# Model performance

The logistic regression model used in the analysis produced a ROC-AUC of 0.930 and PR-AUC 0.750 on held-out test data. The highest risk 10% of retained customers were targeted to streamline business recommendations. The top 10% captured over 49% of historical churners with over 78% precision. Lift was 4.91 over baseline.

# Recommendations

1. Falling activity trend
 Prompt: Meaningful decline in transaction frequency.
 Why: Transaction count is a strong signal. Churned averaged 44.9 transactions vs 68.7 for retained.
 Action: Spend-based incentive or card usage reminder.

2. Repeated service contacts
 Prompt: Higher than typical contact frequency.
 Why: Contact count frequency raised churn odds, suggesting potential unresolved issues.
 Action: A senior service rep can review the case and reach out accordingly to address any potential issues.

3. High Inactivity
 Prompt: High months inactive.
 Why: Churn climbs steeply with inactivity, peaking at 4 months. This presents a window for action.
 Action: Send a timely email or app nudge before 4 months of inactivity.

4. Only 1 to 2 products
 Prompt: Low number of products held.
 Why: Customers with 1 or 2 products demonstrate roughly twice the attrition of those with 5 to 6.
 Action: Recommend additional products, or explore opportunities to deepen relationships with the customer.

Priority: Customers who meet several prompts first.

# Test retention:

The model identifies customers who are at an elevated risk, but it cannot conclude that any intervention will always reduce churn. It is advised to test targeted customers who are receiving recommended actions against a control group to confirm strategy success.

